# Welcome to the 2026 AIMS PINN Workshop — Python example

This short notebook shows you around: where your files live, how to use the shared course materials, how to use and install Python packages, and where to save data.


## Your folders

- **`~`** (your home, e.g. `/home/efs/<you>`) — your **persistent** workspace on EFS. **Save your work here.** It is the *same* on the CPU and GPU hubs, backed up daily, with a **20 GB** quota.
- **`~/course-materials`** — a **read-only** copy of the course GitHub repo ([open-AIMS/Julia_PINN_training_2026](https://github.com/open-AIMS/Julia_PINN_training_2026)), refreshed automatically. To change a file, **copy it into your home first**.
- **`~/my_work`** — (created below) a good place to keep outputs.
- Don't rely on `/tmp` — it is wiped. Don't try to write into `~/course-materials` — it is read-only.


## 0. Where am I?


In [ ]:
import os, sys
print('Home directory :', os.path.expanduser('~'))
print('Working dir    :', os.getcwd())
print('Python         :', sys.version.split()[0])

## 1. The shared course materials

Everything from the course GitHub repo is available read-only at `~/course-materials`.


In [ ]:
import os
materials = os.path.expanduser('~/course-materials')
assert os.path.isdir(materials), 'course-materials not found — it appears on login'
print('Course materials live in:', materials, '\n')
for name in sorted(os.listdir(materials))[:20]:
    print('  ', name)

## 2. Using packages

Scientific packages (numpy, scipy, matplotlib, …) are preinstalled:


In [ ]:
import numpy as np
rng = np.random.default_rng(42)
A = rng.random((3, 3))
print('A =\n', A)
print('\neigenvalues :', np.linalg.eigvals(A))
print('column means:', A.mean(axis=0))

## 3. Installing your own package

`pip install` puts packages in **your** `~/.local` (because `PIP_USER=1`), so they **persist** on EFS across sessions and both hubs.

As an example we install [`distsfactory`](https://pypi.org/project/distsfactory/) — a small package for constructing probability distributions from partial specifications (moments, quantiles, mode, support):

In [ ]:
%pip install --quiet distsfactory

In [ ]:
import distsfactory
print("distsfactory", getattr(distsfactory, "__version__", "installed"))
# what does the package give us?
print("names:", [n for n in dir(distsfactory) if not n.startswith("_")][:12])

## 4. Saving your data

Save anything you want to keep in your **home** — for example the **`~/my_work`** folder (the one the file browser opens into), as the cell below does. Your home is on EFS — persistent + backed up. Mind the 20 GB quota.


In [ ]:
import os
outdir = os.path.expanduser('~/my_work'); os.makedirs(outdir, exist_ok=True)
outfile = os.path.join(outdir, 'hello_python.txt')
with open(outfile, 'w') as f:
    f.write('Hello from Python on the AIMS PINN hub!\n')
print('Saved to :', outfile)
print('Contents :', open(outfile).read())

## 5. GPU check (GPU hub)

The **GPU hub** gives you an NVIDIA GPU — the login page and the JupyterLab top bar show *which one* (it can vary day to day, but your code never changes). **PyTorch** and **JAX** are preinstalled with CUDA support.

The cell below runs small GPU computations when a GPU is present, and explains itself on the CPU hub.

In [ ]:
# GPU check — runs real tests on the GPU hub; explains gracefully on the CPU hub.
import torch
if torch.cuda.is_available():
    print("✅ PyTorch sees GPU:", torch.cuda.get_device_name(0))
    x = torch.randn(4000, 4000, device="cuda")
    print("   GPU matmul OK, sum =", float((x @ x).sum()))
else:
    print("ℹ️  PyTorch: no active GPU here — you're on the CPU hub (or the GPU isn't up).")

try:
    import jax, jax.numpy as jnp
    devs = jax.devices()
    if any(d.platform == "gpu" for d in devs):
        print("✅ JAX sees GPU:", devs)
        xx = jnp.ones((4000, 4000))
        print("   JAX matmul sum =", float((xx @ xx).sum()))
    else:
        print("ℹ️  JAX: running on CPU —", devs)
except Exception as e:
    print("JAX check skipped:", e)

# Same code works on whichever GPU the hub has that day (A10G / L4 / T4).
# Run on the GPU hub (gpu.aims.accumulationpoint.com, 8am–8pm AEST) to use the GPU.

## 6. PINN smoke test — push the machine 🚀

A tiny **physics-informed neural network (PINN) in PyTorch**, solving the 1-D Poisson problem $-u''(x)=\sin(\pi x)$ on $[0,1]$ with $u(0)=u(1)=0$ (exact: $u=\sin(\pi x)/\pi^2$). It trains a small MLP using **autograd-computed derivatives** for the physics loss — a good "does PyTorch train quickly here?" check.

It runs on the CPU, and **automatically uses the GPU if you are on the GPU hub**.

In [ ]:
import torch, time, math
import matplotlib.pyplot as plt

dev = "cuda" if torch.cuda.is_available() else "cpu"
print("training PINN on:", dev)
torch.manual_seed(0)

net = torch.nn.Sequential(
    torch.nn.Linear(1, 32), torch.nn.Tanh(),
    torch.nn.Linear(32, 32), torch.nn.Tanh(),
    torch.nn.Linear(32, 1),
).to(dev)

xf = torch.linspace(0, 1, 128, device=dev).reshape(-1, 1).requires_grad_(True)  # collocation pts
xb = torch.tensor([[0.0], [1.0]], device=dev)                                   # boundary pts

def pde_residual(x):
    u   = net(x)
    ux  = torch.autograd.grad(u,  x, torch.ones_like(u),  create_graph=True)[0]
    uxx = torch.autograd.grad(ux, x, torch.ones_like(ux), create_graph=True)[0]
    return uxx + torch.sin(math.pi * x)        # -u'' = sin(pi x)

opt = torch.optim.Adam(net.parameters(), lr=1e-2)
t0 = time.time()
for it in range(2000):
    opt.zero_grad()
    loss = (pde_residual(xf)**2).mean() + (net(xb)**2).mean()   # physics + boundary
    loss.backward(); opt.step()
print(f"trained 2000 steps in {time.time()-t0:.1f}s   |   final loss = {loss.item():.3e}")

with torch.no_grad():
    xs = torch.linspace(0, 1, 101, device=dev).reshape(-1, 1)
    up = net(xs).cpu().numpy().ravel()
xs_np  = xs.cpu().numpy().ravel()
uexact = torch.sin(math.pi * xs).cpu().numpy().ravel() / math.pi**2
print("max abs error vs exact =", round(float(abs(up - uexact).max()), 5))

plt.plot(xs_np, uexact, label="exact", lw=3)
plt.plot(xs_np, up, "--", label="PINN", lw=2)
plt.legend(); plt.xlabel("x"); plt.ylabel("u(x)"); plt.title("PyTorch PINN: -u'' = sin(pi x)")
plt.show()

---
Happy modelling! Duplicate this notebook (right-click → *Duplicate*), or copy files out of `~/course-materials`, to start experimenting.
